### Phase 0: Task-Adaptive Pre-Training (TAPT)

**Goal:** Domain adaptation - align Longformer's embeddings with political news vocabulary and structure.

**Approach:**
- Base Model: `allenai/longformer-base-4096`
- Architecture: `LongformerForMaskedLM`
- Dataset: ~360k unlabeled news articles
- Objective: Masked Language Modeling (15% dynamic masking)
- Output: `longformer-news-base` weights for Phase 1

**Workflow:** Develop and test here, then convert to script for full training run. Running with like 10,000 examples then moving to script.

## Task 1: Environment Setup & Imports

In [9]:
import os
import torch
import psycopg2
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import (
    LongformerTokenizer,
    LongformerForMaskedLM,
    DataCollatorForLanguageModeling,
)
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

load_dotenv()

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
VRAM: 17.2 GB


## Task 2: Load Article Text from Database

In [10]:
# Load raw article text for TAPT
# We only need maintext - no labels needed for MLM

conn = psycopg2.connect(
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)
cur = conn.cursor()

# Get all articles with text (no labels needed for unsupervised MLM)
cur.execute("""
    SELECT b.maintext
    FROM mm_framing_full a
    JOIN newsarticles b ON a.url = b.url
    WHERE b.maintext IS NOT NULL
    AND LENGTH(b.maintext) > 100
    LIMIT 3000
""")

articles = [row[0] for row in cur.fetchall()]
cur.close()
conn.close()

print(f"Loaded {len(articles):,}")
print(f"Sample length: {len(articles[0]):,} chars")
print(f"Preview: {articles[0][:200]}...")

Loaded 3,000
Sample length: 171 chars
Preview: An eight-week-old puppy is in intensive care after undergoing life-saving surgery. Her body was so badly burned the veterinarian was brought to tears. Ryan Hughes reports....


## Task 3: Create MLM Dataset Class

In [11]:
# Custom Dataset for MLM
# Tokenizes on-the-fly to avoid memory issues with 360k articles

class ArticleMLMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=2048):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Tokenize with truncation
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        # Squeeze batch dimension to make sure we're returning just a single example (DataLoader will re-batch)
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }
        
        # modified from the V2 since we don't need global attention, we only need local attention for next-word token prediction
        

## Task 4: Initialize Tokenizer, Split Data, Create Datasets

In [12]:
# Initialize tokenizer
tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")

# Train/val split (95/5 - maximize training data for TAPT)
train_texts, val_texts = train_test_split(
    articles,
    test_size=0.05,
    random_state=42
)

print(f"Train: {len(train_texts):,} articles")
print(f"Val: {len(val_texts):,} articles")

# Create datasets
train_dataset = ArticleMLMDataset(train_texts, tokenizer, max_length=2048)
val_dataset = ArticleMLMDataset(val_texts, tokenizer, max_length=2048)

# Test one sample
sample = train_dataset[0]
print(f"\nSample input_ids shape: {sample['input_ids'].shape}")
print(f"Sample attention_mask shape: {sample['attention_mask'].shape}")
print(f"Non-padding tokens: {sample['attention_mask'].sum().item()}")

Tokenizer vocab size: 50,265
Train: 2,850 articles
Val: 150 articles

Sample input_ids shape: torch.Size([2048])
Sample attention_mask shape: torch.Size([2048])
Non-padding tokens: 136


## Task 5: Create Data Collator and DataLoaders

In [ ]:
BATCH_SIZE = 8 # Based on testing, GPU can handle size of 8

# Data collator handles dynamic 15% masking
# This is the key to MLM - it randomly masks tokens each batch
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15  # 15% masking rate per spec
)

# Create DataLoaders
# Batch size 1 + gradient accumulation for VRAM management
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0, # workers significantly slows down by needing data copying
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
    pin_memory=True # pin memory for faster async transfer to GPU, minor
)

# Test the collator output
test_batch = next(iter(train_loader))
print(f"Batch keys: {test_batch.keys()}")
print(f"input_ids shape: {test_batch['input_ids'].shape}")
print(f"labels shape: {test_batch['labels'].shape}")

# Verify masking rate
masked_count = (test_batch['labels'] != -100).sum().item()
total_tokens = test_batch['attention_mask'].sum().item()
print(f"Masked tokens: {masked_count} / {total_tokens} ({100*masked_count/total_tokens:.1f}%)")

Batch keys: KeysView({'input_ids': tensor([[    0, 33383,     9,  ...,     1,     1,     1],
        [    0,  3084,   165,  ...,     1,     1,     1],
        [    0,   500,  6368,  ...,     1,     1,     1],
        [    0, 40348, 17597,  ...,     1,     1,     1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100]])})
input_ids shape: torch.Size([4, 2048])
labels shape: torch.Size([4, 2048])
Masked tokens: 265 / 1657 (16.0%)


## Task 6: Initialize Model

In [19]:
# Load Longformer for Masked Language Modeling
model = LongformerForMaskedLM.from_pretrained("allenai/longformer-base-4096")
model.to(device)

# Enable gradient checkpointing (trades compute for VRAM)
model.gradient_checkpointing_enable()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model parameters: 148,711,257
Trainable parameters: 148,711,257


## Task 7: MLM Training Loop [YOU IMPLEMENT]

This is the core of TAPT. Implement the training loop that:
1. Runs for N epochs (start with 1 for testing)
2. Uses gradient accumulation (effective batch size 16)
3. Computes MLM loss and backpropagates
4. Tracks and logs loss

**Key concepts:**
- The DataCollator already set up `labels` with -100 for non-masked tokens
- The model's `forward()` computes cross-entropy loss only on masked positions
- Just call: `outputs = model(**batch)` and `loss = outputs.loss`

**Hyperparameters to consider:**
- `NUM_EPOCHS` = 10 (use 1 for testing)
- `GRAD_ACCUM_STEPS` = 16
- `LEARNING_RATE` = 5e-5
- Mixed precision with `torch.cuda.amp`

Key Innovations / Notes:

* We use a learning rate schedule with warmup, this slowly increases the LR from 0 to a target value, then linearly decreases it to 0. It can be used from the transformers library `get_linear_schedule_with_warmup()`

In [21]:
# FOR testing have a clean slate on GPU
import gc
gc.collect()
torch.cuda.empty_cache() 

In [20]:
# Implement MLM training loop
from transformers import get_linear_schedule_with_warmup

# Hyperparameters
NUM_EPOCHS = 1  # Start with 1 for testing
GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 5e-5

# Parameters for LR scheduler
num_training_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
num_warmup_steps = int(0.1 * num_training_steps)  # usually 10% of total steps

print(f"Training steps: {num_training_steps:,}")
print(f"Warmup steps: {num_warmup_steps:,}")

# 1. Create optimizer (AdamW)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

# 2. Create learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# 3. Mixed precision setup - GradScaler for fp16 training
scaler = torch.cuda.amp.GradScaler()

# 4. Training loop with gradient accumulation
for epoch in range(NUM_EPOCHS):
    model.train()  # enables dropout, called once per epoch
    total_loss = 0
    optimizer.zero_grad()  # zero gradients at start of epoch
    
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}")
    
    for step, batch in progress_bar:
        # Move batch to CUDA
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass with mixed precision
        # this 'autocast' method is purely a performance optimization
        with torch.cuda.amp.autocast():
            outputs = model(**batch) # batch includes labels, which is the token identifier - many labels!
            loss = outputs.loss
            loss = loss / GRAD_ACCUM_STEPS  # normalize loss for accumulation
        
        # Backward pass with scaler (handles fp16 gradients)
        scaler.scale(loss).backward()
        
        total_loss += loss.item() * GRAD_ACCUM_STEPS  # track un-normalized loss
        
        # Update weights every GRAD_ACCUM_STEPS
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)  # clear gradients for next accumulation
            
            # Log progress
            avg_loss = total_loss / (step + 1)
            current_lr = scheduler.get_last_lr()[0]
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{current_lr:.2e}"})
    
    # End of epoch stats
    epoch_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete. Average loss: {epoch_loss:.4f}")
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast():
                outputs = model(**batch)
                val_loss += outputs.loss.item()
    
    val_loss /= len(val_loader)
    print(f"Validation loss: {val_loss:.4f}")

C:\Users\rhrou\AppData\Local\Temp\ipykernel_7608\2484719343.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Training steps: 44
Warmup steps: 4


Epoch 1:   0%|          | 0/713 [00:00<?, ?it/s]C:\Users\rhrou\AppData\Local\Temp\ipykernel_7608\2484719343.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1:   4%|▍         | 28/713 [00:23<09:37,  1.19it/s, loss=1.6037, lr=1.25e-05]


KeyboardInterrupt: 

## Task 8: Save Adapted Model [YOU IMPLEMENT]

After training, save the model for Phase 1.

**Save location:** `saved_models/longformer-news-base/`

**What to save:**
- Model weights
- Tokenizer
- (Optional) Training config for reproducibility

In [ ]:
# Save the domain-adapted model

SAVE_PATH = "saved_models/longformer-news-base"

# Create directory if needed
os.makedirs(SAVE_PATH, exist_ok=True)

# 1. Save model weights
model.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

# 2. Save tokenizer
tokenizer.save_pretrained(SAVE_PATH)
print(f"Tokenizer saved to {SAVE_PATH}")

# 3. Verify it loads correctly
print("\nVerifying saved model loads...")
test_model = LongformerForMaskedLM.from_pretrained(SAVE_PATH)
test_tokenizer = LongformerTokenizer.from_pretrained(SAVE_PATH)
print("Model and tokenizer loaded successfully!")

# Quick sanity check - model should predict reasonable words
test_text = "The senator announced new <mask> on climate change."
inputs = test_tokenizer(test_text, return_tensors="pt")

test_model.eval()
with torch.no_grad():
    outputs = test_model(**inputs)
    mask_idx = (inputs["input_ids"] == test_tokenizer.mask_token_id).nonzero()[0, 1]
    probs = outputs.logits[0, mask_idx].softmax(dim=-1)
    top5 = probs.topk(5)

print(f"\nTop 5 predictions for '{test_text}':")
for prob, idx in zip(top5.values, top5.indices):
    print(f"  {test_tokenizer.decode([idx])}: {prob:.3f}")

del test_model  # Free memory
print("\nTAPT complete! Model ready for Phase 1 fine-tuning.")